# Praktikum 3 - Splitting Data (Data Latih, Validasi, dan Uji)

**Mata Kuliah:** Pembelajaran Mesin
**Jobsheet 03:** Ekstraksi Fitur

### Tujuan Praktikum:
1. Menerapkan strategi **Random Split** (Rasio 8:1:1).
2. Menerapkan strategi **Stratified Split** (Rasio 8:1:1) dengan mempertahankan rasio kelas target.
3. Menerapkan metode **K-Fold Cross Validation** dan **Stratified K-Fold Cross Validation**.

## 1. Random Split (Rasio 8:1:1)

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

data_path = r'..\Dataset\Titanic-Dataset-selected.csv'

df = pd.read_csv(data_path)
print('Dimensi data:', df.shape)
df.head()

Dimensi data: (891, 5)


,Survived,Pclass,Age,Sex,Cabin
0,0,3,-0.592481,1,115
1,1,1,0.638789,0,81
2,1,3,-0.284663,0,115
3,1,1,0.407926,0,55
4,0,3,0.407926,1,115


In [2]:
# Split data training (80%) dan unseen (20%)
df_train, df_unseen = train_test_split(df, test_size=0.2, random_state=0)

# Split unseen menjadi validasi (10%) dan testing (10%)
df_val, df_test = train_test_split(df_unseen, test_size=0.5, random_state=0)

print(f'Jumlah data asli: {df.shape[0]}')
print(f'Jumlah data train: {df_train.shape[0]}')
print(f'Jumlah data val: {df_val.shape[0]}')
print(f'Jumlah data test: {df_test.shape[0]}')
print('=========')
print(f'Distribusi label asli:\n{df.Survived.value_counts()}')
print(f'Distribusi label train:\n{df_train.Survived.value_counts()}')
print(f'Distribusi label val:\n{df_val.Survived.value_counts()}')
print(f'Distribusi label test:\n{df_test.Survived.value_counts()}')

Jumlah data asli: 891
Jumlah data train: 712
Jumlah data val: 89
Jumlah data test: 90
Distribusi label asli:
Survived
0    549
1    342
Name: count, dtype: int64
Distribusi label train:
Survived
0    439
1    273
Name: count, dtype: int64
Distribusi label val:
Survived
0    53
1    36
Name: count, dtype: int64
Distribusi label test:
Survived
0    57
1    33
Name: count, dtype: int64


## 2. Stratified Split (Rasio 8:1:1)

In [3]:
# Stratified split menjaga proporsi label Survived di setiap subset
df2 = df.copy()
df2_train, df2_unseen = train_test_split(df2, test_size=0.2, random_state=0, stratify=df2['Survived'])
df2_val, df2_test = train_test_split(df2_unseen, test_size=0.5, random_state=0, stratify=df2_unseen['Survived'])

print('=== Hasil Stratified Split ===')
print(f'Distribusi label train:\n{df2_train.Survived.value_counts()}')
print(f'Distribusi label val:\n{df2_val.Survived.value_counts()}')
print(f'Distribusi label test:\n{df2_test.Survived.value_counts()}')

=== Hasil Stratified Split ===
Distribusi label train:
Survived
0    439
1    273
Name: count, dtype: int64
Distribusi label val:
Survived
0    55
1    34
Name: count, dtype: int64
Distribusi label test:
Survived
0    55
1    35
Name: count, dtype: int64


## 3. K-Fold Cross Validation

In [4]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=4, shuffle=True, random_state=42)
print(f'Jumlah fold: {kf.get_n_splits()}')

for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
    print(f'Fold {fold+1} | Train samples: {len(train_idx)}, Test samples: {len(test_idx)}')
    print(f'   Test indices: {test_idx[:10]} ...')

Jumlah fold: 4
Fold 1 | Train samples: 668, Test samples: 223
   Test indices: [ 5  7 10 23 25 29 30 31 33 39] ...
Fold 2 | Train samples: 668, Test samples: 223
   Test indices: [ 2  3  6  9 11 12 15 18 22 24] ...
Fold 3 | Train samples: 668, Test samples: 223
   Test indices: [ 0  8 16 17 19 26 36 37 38 45] ...
Fold 4 | Train samples: 669, Test samples: 222
   Test indices: [ 1  4 13 14 20 21 27 32 34 35] ...


## 4. Stratified K-Fold Cross Validation

In [5]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['Survived'])):
    surv_test = df.iloc[test_idx]['Survived'].value_counts().to_dict()
    print(f'Fold {fold+1} | Total test: {len(test_idx)} | Label ratio in test: {surv_test}')

Fold 1 | Total test: 223 | Label ratio in test: {0: 138, 1: 85}
Fold 2 | Total test: 223 | Label ratio in test: {0: 137, 1: 86}
Fold 3 | Total test: 223 | Label ratio in test: {0: 137, 1: 86}
Fold 4 | Total test: 222 | Label ratio in test: {0: 137, 1: 85}
